##Step 1: Import Libraries

In [51]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [52]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/clean_loan_dataset.csv')

In [53]:
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.822149    -0.653722        -0.978632    0.246080              1   
1  0.564574    -0.737864         0.209402   -0.458384              1   
2 -0.856268     0.000000        -0.611111   -0.414958              0   
3 -0.911156    -0.498382         0.431624   -0.661037              1   
4 -0.649046    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.445244               8.274536  
1                 0         1     -0.912749               0.153994  
2                 0         1      0.280113              -0.126321  
3                 0         1     -0.315175              -0.669033  
4                 0         1     -0.284688              -0.284975  


In [54]:
print(df.shape)

(100, 9)


#2)Prepare Features and Target

In [55]:
X = df.drop(columns=["Approved"])
y = df["Approved"]

#3) Train/test split

In [56]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (80, 8)
Testing shape: (20, 8)


### 4) Train classifiers


In [57]:
RANDOM_STATE = 42
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=1000, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
svc = SVC(kernel="rbf", random_state=RANDOM_STATE)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)
dt.fit(X_train, y_train)
gb.fit(X_train, y_train)
svc.fit(X_train, y_train)

SVC(random_state=42)

### 5) Helper functions: metrics + confusion matrix

In [58]:
def print_clf_metrics(name, y_true, y_pred, pos_label=1):
    """Print Accuracy, Precision, Recall, F1. pos_label=1 means Approved is positive."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    print(f"\n{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (positive = Approved=1)")
    print(f"  Recall   : {rec:.3f}  (positive = Approved=1)")
    print(f"  F1-Score : {f1:.3f}  (positive = Approved=1)")


def print_confmat(name, y_true, y_pred):
    """
    Confusion matrix with readable labels.
    Rows = Actual, Cols = Predicted
    Order: [Approved(1), Rejected(0)]
    """
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: Approved (1)", "Actual: Rejected (0)"],
        columns=["Pred: Approved (1)", "Pred: Rejected (0)"],
    )
    print(f"\n{name} – Confusion Matrix:\n{cm_df}")


def label_str(v):
    return "Approved (1)" if v == 1 else "Rejected (0)"

### 6) Evaluate Performance

In [61]:

models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "Decision Tree": dt,
    "Gradient Boosting": gb,
    "SVM": svc
}

results = {}

for name, model in models.items():
    preds = model.predict(X_test)
    print_clf_metrics(name, y_test, preds, pos_label=1)
    print_confmat(name, y_test, preds)
    results[name] = accuracy_score(y_test, preds)


Logistic Regression Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Logistic Regression – Confusion Matrix:
                      Pred: Approved (1)  Pred: Rejected (0)
Actual: Approved (1)                  11                   2
Actual: Rejected (0)                   4                   3

Random Forest Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Random Forest – Confusion Matrix:
                      Pred: Approved (1)  Pred: Rejected (0)
Actual: Approved (1)                  11                   2
Actual: Rejected (0)                   4                   3

Decision Tree Performance:
  Accuracy : 0.700
  Precision: 0.706  (positive = Approved=1)
  Recall   : 0.923  (positive = Approved=1)
  F1-Score : 0.800  (positive = Approved=1)

Decision Tree 

# 7) Single-row sanity check

In [63]:
# 7) Single-row sanity check
# You can change the index 'i' to test different rows from the X_test dataset.
i = 0 # Example: picking the first row from X_test
x_one_df = X_test.iloc[[i]]
y_true = y_test.iloc[i]

print("\n=== SINGLE APPLICATION CHECK ===")
print(f"  Actual label : {label_str(y_true)}")

# Print predictions from each model
print(f"  Logistic Regression   : {label_str(int(log_reg.predict(x_one_df)[0]))}")
print(f"  Random Forest         : {label_str(int(rf.predict(x_one_df)[0]))}")
print(f"  Decision Tree         : {label_str(int(dt.predict(x_one_df)[0]))}")
print(f"  Gradient Boosting     : {label_str(int(gb.predict(x_one_df)[0]))}")
print(f"  SVM                   : {label_str(int(svc.predict(x_one_df)[0]))}")

print("\nSingle row data used for prediction:")
display(x_one_df)


=== SINGLE APPLICATION CHECK ===
  Actual label : Approved (1)
  Logistic Regression   : Rejected (0)
  Random Forest         : Rejected (0)
  Decision Tree         : Approved (1)
  Gradient Boosting     : Approved (1)
  SVM                   : Rejected (0)

Single row data used for prediction:


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,DebtToIncome,IncomePerYearEmployed
70,0.971475,-0.517799,0.448718,2.196019,0,0,0.844548,0.178435
